In [3]:
%pip install astroquery astropy plotly pandas

   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.1 MB 6.9 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/11.1 MB 2.9 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.1 MB 2.9 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.1 MB 1.6 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/11.1 MB 1.3 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/11.1 MB 1.3 MB/s eta 0:00:08
   ------ --------------------------------- 1.8/11.1 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.1/11.1 MB 1.2 MB/s eta 0:00:08
   -------- ------------------------------- 2.4/11.1 MB 1.2 MB/s eta 0:00:08
   --------- ------------------------------ 2.6/11.1 MB 1.2 MB/s eta 0:00:07
   ---------- ----------------------------- 2.9/11.1 MB 1.2 MB/s eta 0:00:07
   ----------- ---------------------------- 3.1/11.1 MB 1.2 MB/s eta 0:00:07
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
!pip install astropy plotly pandas

You should consider upgrading via the 'c:\users\nisha sharma\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from astropy.coordinates import SkyCoord
import astropy.units as u

print("🎲 Generating a local mock star dataset instantly...")

# Create 1,000 random stars locally to bypass the slow SIMBAD server
np.random.seed(42)
num_stars = 1000

# Generate realistic random star data
mock_names = [f"Star-{i}" for i in range(num_stars)]
mock_ra = np.random.uniform(0, 360, num_stars)      # RA degrees
mock_dec = np.random.uniform(-90, 90, num_stars)    # Dec degrees
mock_flux = np.random.uniform(-1.46, 6.0, num_stars) # Magnitudes (from Sirius out to limit)

df = pd.DataFrame({
    'star_name': mock_names,
    'ra': mock_ra,
    'dec': mock_dec,
    'magnitude': mock_flux
})

print("📐 Mapping coordinates onto a 3D Celestial Sphere...")
# Map the coordinates locally
coords = SkyCoord(ra=df['ra'].values, dec=df['dec'].values, unit=(u.deg, u.deg), distance=1*u.pc)

df['x'] = coords.cartesian.x
df['y'] = coords.cartesian.y
df['z'] = coords.cartesian.z

# Dynamically size stars so bright stars appear larger
df['display_size'] = ((6.0 - df['magnitude']) / 2) + 2

print("🌌 Rendering Interactive Plot...")
# Create the interactive 3D plot
fig = go.Figure(data=[go.Scatter3d(
    x=df['x'],
    y=df['y'],
    z=df['z'],
    mode='markers',
    marker=dict(
        size=df['display_size'],
        color=df['magnitude'], 
        colorscale='Hot',  
        opacity=0.8,
        reversescale=True  
    ),
    hovertext=df['star_name'],
    hoverinfo='text'
)])

# Style the plot for a deep-space look
fig.update_layout(
    title=f"Global 3D Celestial Sphere Map ({len(df)} Simulated Brightest Stars)",
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    template="plotly_dark"
)

# Render inside Jupyter Lab / Notebook
fig.show()

🎲 Generating a local mock star dataset instantly...
📐 Mapping coordinates onto a 3D Celestial Sphere...
🌌 Rendering Interactive Plot...


In [4]:
from astroquery.jplhorizons import Horizons
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
import pandas as pd
import plotly.graph_objects as go

print("🪐 Fetching real-time planet positions from NASA JPL...")

# 1. Fetch planet coordinates from NASA using a Julian Date
jd_today = Time.now().jd
planet_ids = {'Venus': '299', 'Mars': '499', 'Jupiter': '599'}
planet_data = []

for name, pid in planet_ids.items():
    obj = Horizons(id=pid, location='500', epochs=jd_today)
    el = obj.ephemerides()
    
    ra_val = el['RA'][0]
    dec_val = el['DEC'][0]
    
    p_coord = SkyCoord(ra=ra_val*u.deg, dec=dec_val*u.deg, distance=1*u.pc)
    
    planet_data.append({
        'name': name,
        'x': p_coord.cartesian.x.value,
        'y': p_coord.cartesian.y.value,
        'z': p_coord.cartesian.z.value
    })

df_planets = pd.DataFrame(planet_data)

# FIX: Define 'fig' explicitly so Python recognizes it!
fig = go.Figure()

# 2. Add the planets as a bright layer
fig.add_trace(go.Scatter3d(
    x=df_planets['x'],
    y=df_planets['y'],
    z=df_planets['z'],
    mode='markers+text',
    text=df_planets['name'],
    textposition="top center",
    marker=dict(
        size=10,
        color='cyan',  # Neon cyan color
        symbol='diamond', 
        line=dict(color='white', width=1)
    ),
    hoverinfo='text'
))

# 3. Apply a clean dark theme layout matching your cosmic theme
fig.update_layout(
    title="Solar System Planet Positions",
    template="plotly_dark",
    scene=dict(
        xaxis=dict(title='X (pc)', range=[-1.5, 1.5]),
        yaxis=dict(title='Y (pc)', range=[-1.5, 1.5]),
        zaxis=dict(title='Z (pc)', range=[-1.5, 1.5])
    )
)

print("✅ Solar system layer successfully injected!")
fig.show()

🪐 Fetching real-time planet positions from NASA JPL...
✅ Solar system layer successfully injected!


In [10]:
from astroquery.simbad import Simbad
from astroquery.jplhorizons import Horizons
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
import pandas as pd
import plotly.graph_objects as go
import numpy as np

print("✨ Step 1: Querying SIMBAD database for bright stars...")
custom_simbad = Simbad()
# FIX: Use 'flux' as explicitly demanded by the new astroquery API version
custom_simbad.add_votable_fields('flux')

# Fetch stars brighter than magnitude 4.0
try:
    star_table = custom_simbad.query_criteria('Vmag < 4.0')
except Exception as e:
    print(f"⚠️ SIMBAD query failed, trying fallback method... Error: {e}")
    star_table = custom_simbad.query_catalog("I/239/hip_main")

star_data = []

if star_table is not None:
    df_raw = star_table.to_pandas()
    
    # Locate columns safely
    ra_col = [c for c in df_raw.columns if 'ra' in c.lower()][0]
    dec_col = [c for c in df_raw.columns if 'dec' in c.lower()][0]
    mag_col = [c for c in df_raw.columns if 'flux_v' in c.lower() or 'vmag' in c.lower()]
    mag_key = mag_col[0] if mag_col else None
    name_key = [c for c in df_raw.columns if 'main_id' in c.lower() or 'hip' in c.lower()][0]

    for _, row in df_raw.iterrows():
        try:
            if pd.isna(row[ra_col]) or pd.isna(row[dec_col]):
                continue
                
            ra_val = row[ra_col]
            dec_val = row[dec_col]
            
            name = ra_val.decode('utf-8') if isinstance(row[name_key], bytes) else str(row[name_key])
            mag = float(row[mag_key]) if (mag_key and not pd.isna(row[mag_key])) else 2.0
            
            if isinstance(ra_val, (int, float, np.number)):
                coord = SkyCoord(ra=ra_val*u.deg, dec=dec_val*u.deg, distance=1*u.pc)
            else:
                ra_str = ra_val.decode('utf-8') if isinstance(ra_val, bytes) else str(ra_val)
                dec_str = dec_val.decode('utf-8') if isinstance(dec_val, bytes) else str(dec_val)
                coord = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg), distance=1*u.pc)
            
            star_data.append({
                'name': name,
                'x': coord.cartesian.x.value,
                'y': coord.cartesian.y.value,
                'z': coord.cartesian.z.value,
                'mag': mag
            })
        except Exception:
            continue

df_stars = pd.DataFrame(star_data)

if df_stars.empty:
    print("⚠️ Warning: No stars parsed from database. Generating reference grid stars.")
    df_stars = pd.DataFrame([
        {'name': 'Sirius Reference', 'x': 0.5, 'y': 0.5, 'z': 0.5, 'mag': 1.4},
        {'name': 'Vega Reference', 'x': -0.5, 'y': 0.5, 'z': -0.3, 'mag': 0.0}
    ])

print(f"✅ Successfully processed {len(df_stars)} stars.")

print("🪐 Step 2: Querying NASA JPL for live planet positions...")
planet_data = []
try:
    jd_today = Time.now().jd
    planet_ids = {'Venus': '299', 'Mars': '499', 'Jupiter': '599'}
    
    for name, pid in planet_ids.items():
        obj = Horizons(id=pid, location='500', epochs=jd_today)
        el = obj.ephemerides()
        p_coord = SkyCoord(ra=el['RA'][0]*u.deg, dec=el['DEC'][0]*u.deg, distance=1*u.pc)
        
        planet_data.append({
            'name': name,
            'x': p_coord.cartesian.x.value,
            'y': p_coord.cartesian.y.value,
            'z': p_coord.cartesian.z.value
        })
except Exception as e:
    print(f"⚠️ NASA JPL API failed. Using historical positions. Error: {e}")
    planet_data = [
        {'name': 'Venus', 'x': 0.1, 'y': -0.8, 'z': -0.3},
        {'name': 'Mars', 'x': 0.7, 'y': 0.2, 'z': 0.4},
        {'name': 'Jupiter', 'x': -0.4, 'y': 0.6, 'z': 0.1}
    ]

df_planets = pd.DataFrame(planet_data)

print("🎨 Step 3: Generating interactive 3D Celestial Map...")
fig = go.Figure()

# Layer A: The Background Stars
fig.add_trace(go.Scatter3d(
    x=df_stars['x'], y=df_stars['y'], z=df_stars['z'],
    mode='markers',
    text=df_stars['name'],
    marker=dict(
        size=np.clip(8 - df_stars['mag'], 2, 12),
        color=df_stars['mag'],
        colorscale='Viridis',
        showscale=False,
        line=dict(color='black', width=0.5)
    ),
    name='Stars',
    hoverinfo='text'
))

# Layer B: The Planets
fig.add_trace(go.Scatter3d(
    x=df_planets['x'], y=df_planets['y'], z=df_planets['z'],
    mode='markers+text',
    text=df_planets['name'],
    textposition="top center",
    marker=dict(
        size=11,
        color='cyan',
        symbol='diamond',
        line=dict(color='white', width=1)
    ),
    name='Planets',
    hoverinfo='text'
))

fig.update_layout
fig.show()

✨ Step 1: Querying SIMBAD database for bright stars...


✅ Successfully processed 4272 stars.
🪐 Step 2: Querying NASA JPL for live planet positions...
🎨 Step 3: Generating interactive 3D Celestial Map...
